<a href="https://colab.research.google.com/github/dipanshu-bot/Dipanshu_CEC/blob/main/Dipanshu_CEC_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap rasterio xarray scipy matplotlib numpy pandas
!pip install git+https://github.com/TUDelftGeodesy/cmod5n.git
!pip install torch torchvision

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project="dipanshucec")

In [ ]:
import ee

aoi = ee.Geometry.Polygon([
[
[68.2088198,24.1951065],
[67.7225872,23.2835917],
[68.5541709,20.1904311],
[71.1446861,19.6992045],
[73.6185503,20.7641857],
[73.4034484,23.0783287],
[72.2043888,23.5573246],
[70.7259080,24.4218284],
[68.2088198,24.1951065]
]
])

start_date='2026-05-04'
end_date='2026-06-04'

In [ ]:
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(aoi)
      .filterDate(start_date,end_date)
      .filter(ee.Filter.eq('instrumentMode','IW'))
      .filter(ee.Filter.listContains(
          'transmitterReceiverPolarisation','VV'))
      .select('VV'))

print("Images:",s1.size().getInfo())

In [ ]:
s1_img = s1.median().clip(aoi)

In [ ]:
vv_db = s1_img.select('VV')

In [ ]:
Map = geemap.Map()

Map.addLayer(
    vv_db,
    {'min':-25,'max':0},
    'VV dB'
)

Map.centerObject(aoi,7)
Map

In [ ]:
water = ee.Image(
    "JRC/GSW1_4/GlobalSurfaceWater"
).select('occurrence')

sea_mask = water.gt(90)

vv_sea = vv_db.updateMask(sea_mask)

In [ ]:
Map = geemap.Map()

Map.addLayer(
    vv_sea,
    {'min':-25,'max':0},
    'Sea only'
)

Map.centerObject(aoi,7)
Map

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=vv_sea,
    description='gujarat_s1',
    folder='SAR',
    fileNamePrefix='gujarat_s1',
    region=aoi,
    scale=10,
    maxPixels=1e13
)

task.start()

In [ ]:
import rasterio
import numpy as np

with rasterio.open("dipanshu_gujarat.tif") as src:
    vv = src.read(1)

vv = np.nan_to_num(vv)

In [ ]:
from scipy.ndimage import median_filter

vv_filtered = median_filter(vv,size=5)

In [ ]:
import torch
from torchvision import models

# Load the pretrained ResNet18 model
model = models.resnet18(pretrained=True)

# Save the original in_features for the fc layer before modification
num_ftrs = model.fc.in_features

# Load the state dictionary from the file
state_dict = torch.load("resnet_wind_direction.pth")

# Create a new state_dict by removing the 'fc' layer keys from the loaded checkpoint.
# This is because the checkpoint's 'fc' layer (with 1000 outputs as per the error)
# is incompatible with our desired 360 output features.
new_state_dict = {k: v for k, v in state_dict.items() if "fc" not in k}

# Load the filtered state_dict into the model.
# `strict=False` is used here to safely ignore any non-matching keys that might remain,
# though the 'fc' layer keys have already been explicitly filtered out.
# This step loads all layers *except* the final fully connected layer.
model.load_state_dict(new_state_dict, strict=False)

# Now, redefine the fc layer with the desired number of output features (360).
# This new layer will be randomly initialized and will need to be trained.
model.fc = torch.nn.Linear(num_ftrs, 360)

model.eval()

In [ ]:
patch_size = 224

patch = vv_filtered[
    1000:1224,
    1000:1224
]

In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224,224))
])

# Convert the 1-channel patch to a 3-channel image by repeating the channel
x = transform(patch).repeat(3, 1, 1).unsqueeze(0)

with torch.no_grad():
    pred = model(x)

wind_dir = pred.argmax().item()

print(wind_dir)

In [ ]:
import torch

checkpoint = torch.load(
    "/content/resnet_wind_direction.pth",
    map_location="cpu"
)

print(type(checkpoint))

if isinstance(checkpoint, dict):
    print(checkpoint.keys())

In [ ]:
state_dict = torch.load(
    "/content/resnet_wind_direction.pth",
    map_location="cpu"
)

print(state_dict['fc.weight'].shape)
print(state_dict['fc.bias'].shape)

In [ ]:
Map.addLayer(
    vv_sea,
    {'min':-25,'max':0},
    'Sea Masked VV'
)